# einops-einsum — faded example 2: Batched vector-matrix product (shared matrix)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-einsum`. Running the beacon reports progress on the `Einops: Deep Learning` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import einsum

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Deep Learning` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-einsum`**, which bridges to the bank subtopic `Einops: Deep Learning` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-einsum"
DD_SUBTOPIC = "Einops: Deep Learning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Concept

Omitting the batch index from an operand broadcasts that operand across the batch. Here a batch of row-vectors `x` (shape `(b, i)`) multiplies a single shared matrix `M` (shape `(i, j)`) whose index string carries no `b`, contracting the shared `i` and preserving `b` and `j`.

## Faded exercise 2

### Faded — batched vector × shared matrix

Implement `batched_vecmat(x, M)` where `x` is `(b, i)` (a batch of row vectors) and `M` is a single `(i, j)` matrix shared across the batch. Output: `(b, j)` with `out[b, j] = sum_i x[b, i] * M[i, j]`.

The surrounding function is given; fill in the single `einops.einsum` call. `M`'s index string must omit `b` so it broadcasts across the batch. Equivalent to `x @ M`.

**Fill in:** the einsum call contracting i between batched x ('b i') and the shared matrix M ('i j'), producing 'b j' so M broadcasts over the batch.

In [ ]:
def batched_vecmat(x: Tensor, M: Tensor) -> Tensor:
    result = None  # TODO: einsum contracting i between x ('b i') and shared M ('i j') -> 'b j'
    return result


def _test():
    t.manual_seed(2)
    x = t.randn(5, 4)
    M = t.randn(4, 7)
    out = batched_vecmat(x, M)
    ref = x @ M
    assert tuple(out.shape) == (5, 7), f'expected (5, 7), got {tuple(out.shape)}'
    assert t.allclose(out, ref, atol=1e-5), 'result does not match x @ M'


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def batched_vecmat(x: Tensor, M: Tensor) -> Tensor:
    result = einsum(x, M, 'b i, i j -> b j')
    return result
```
</details>